# Exp 027c: exp_015d Native Student Blend Benchmark

Benchmark whether a soundscape-only native student distilled from `exp_015d` adds complementary signal on top of the fixed teacher.


In [1]:
from __future__ import annotations

import json
import typing as tp
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def resolve_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'PROJECT_STATE.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not resolve repository root')


@dataclass
class Config:
    experiment_id: str = 'exp_027c'
    experiment_name: str = 'exp015d_native_student_blend_benchmark'
    teacher_experiment: str = 'exp_027a_exp015d_teacher_cache'
    student_experiment: str = 'exp_027b_hgnetv2_soundscape_distill_from_exp015d'
    teacher_dir_override: str | None = None
    student_dir_override: str | None = None
    weight_grid: tuple[float, ...] = tuple(round(x, 2) for x in np.linspace(0.0, 1.0, 21))


CFG = Config()
ROOT = resolve_repo_root()
DATA = ROOT / 'data' / 'birdclef-2026'
OUTPUT_DIR = ROOT / 'experiments' / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


print({'root': str(ROOT), 'output_dir': str(OUTPUT_DIR)})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_027c_exp015d_native_student_blend_benchmark'}


In [2]:
def has_teacher_cache(path: Path) -> bool:
    return (path / 'teacher_meta.parquet').exists() and (path / 'teacher_outputs.npz').exists()


def has_student_outputs(path: Path) -> bool:
    fold_dirs = [p for p in path.iterdir() if p.is_dir() and p.name.startswith('fold_')] if path.exists() else []
    return any((fold_dir / 'best_valid_meta.csv').exists() and (fold_dir / 'best_valid_outputs.npz').exists() for fold_dir in fold_dirs)


def resolve_teacher_dir() -> Path:
    candidates = []
    if CFG.teacher_dir_override:
        candidates.append(Path(CFG.teacher_dir_override).expanduser())
    candidates.extend([
        ROOT / 'experiments' / 'outputs' / CFG.teacher_experiment,
        Path.home() / 'Downloads' / CFG.teacher_experiment,
    ])
    for candidate in candidates:
        if has_teacher_cache(candidate):
            return candidate
    for search_root in [ROOT / 'experiments' / 'outputs', Path.home() / 'Downloads']:
        if not search_root.exists():
            continue
        for meta_path in search_root.rglob('teacher_meta.parquet'):
            parent = meta_path.parent
            if has_teacher_cache(parent) and CFG.teacher_experiment in str(parent):
                return parent
    raise FileNotFoundError(
        'Could not resolve exp_027a teacher cache for exp_027c. '
        'Run exp_027a first or set CFG.teacher_dir_override.'
    )


def resolve_student_dir() -> Path:
    candidates = []
    if CFG.student_dir_override:
        candidates.append(Path(CFG.student_dir_override).expanduser())
    candidates.extend([
        ROOT / 'experiments' / 'outputs' / CFG.student_experiment,
        Path.home() / 'Downloads' / CFG.student_experiment,
    ])
    for candidate in candidates:
        if has_student_outputs(candidate):
            return candidate
    for search_root in [ROOT / 'experiments' / 'outputs', Path.home() / 'Downloads']:
        if not search_root.exists():
            continue
        for fold_meta in search_root.rglob('best_valid_meta.csv'):
            parent = fold_meta.parent.parent
            if has_student_outputs(parent) and CFG.student_experiment in str(parent):
                return parent
    raise FileNotFoundError(
        'Could not resolve exp_027b student outputs for exp_027c. '
        'Run exp_027b first or set CFG.student_dir_override.'
    )


teacher_dir = resolve_teacher_dir()
student_dir = resolve_student_dir()

taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
classes = taxonomy['primary_label'].astype(str).tolist()
class_name_map = taxonomy.set_index('primary_label')['class_name'].to_dict()

teacher_meta = pd.read_parquet(teacher_dir / 'teacher_meta.parquet')
teacher_arr = np.load(teacher_dir / 'teacher_outputs.npz')
teacher_probs = teacher_arr['teacher_probs'].astype(np.float32)
teacher_labels = teacher_arr['labels'].astype(np.float32)

rows_meta = []
rows_teacher = []
rows_student = []
rows_labels = []

fold_dirs = sorted(path for path in student_dir.iterdir() if path.is_dir() and path.name.startswith('fold_'))
if not fold_dirs:
    raise FileNotFoundError(f'No student fold outputs found in {student_dir}')

for fold_dir in fold_dirs:
    meta_student = pd.read_csv(fold_dir / 'best_valid_meta.csv')
    arr_student = np.load(fold_dir / 'best_valid_outputs.npz')
    probs_student = arr_student['probs'].astype(np.float32)
    labels_student = arr_student['labels'].astype(np.float32)

    teacher_fold = teacher_meta.merge(
        meta_student[['row_id']],
        on='row_id',
        how='inner',
    )
    teacher_idx = teacher_fold.index.to_numpy()
    merged = meta_student[['row_id']].merge(
        teacher_meta.reset_index()[['index', 'row_id']],
        on='row_id',
        how='inner',
    )
    assert len(merged) == len(meta_student), (len(merged), len(meta_student))
    teacher_index = merged['index'].to_numpy()

    rows_meta.append(meta_student.assign(fold_src=fold_dir.name))
    rows_teacher.append(teacher_probs[teacher_index])
    rows_student.append(probs_student)
    rows_labels.append(labels_student)

meta_all = pd.concat(rows_meta, ignore_index=True)
teacher_all = np.concatenate(rows_teacher, axis=0)
student_all = np.concatenate(rows_student, axis=0)
labels_all = np.concatenate(rows_labels, axis=0)

assert meta_all.shape[0] == teacher_all.shape[0] == student_all.shape[0] == labels_all.shape[0]

texture_idx = np.array(
    [i for i, label in enumerate(classes) if class_name_map.get(label) in {'Amphibia', 'Insecta'}],
    dtype=np.int64,
)

print({
    'rows': int(len(meta_all)),
    'files': int(meta_all['filename'].nunique()),
    'folds': sorted(meta_all['fold_src'].unique().tolist()),
    'texture_classes': int(len(texture_idx)),
})


{'rows': 168, 'files': 14, 'folds': ['fold_00'], 'texture_classes': 63}


In [3]:
def macro_auc_skip_empty(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int]:
    auc_values: list[float] = []
    for idx in range(y_true.shape[1]):
        yt = y_true[:, idx]
        if len(np.unique(yt)) < 2:
            continue
        auc_values.append(float(roc_auc_score(yt, y_score[:, idx])))
    return float(np.mean(auc_values)), int(len(auc_values))


teacher_macro_auc, teacher_scored = macro_auc_skip_empty(labels_all, teacher_all)
student_macro_auc, student_scored = macro_auc_skip_empty(labels_all, student_all)
teacher_texture_auc, teacher_texture_scored = macro_auc_skip_empty(labels_all[:, texture_idx], teacher_all[:, texture_idx])
student_texture_auc, student_texture_scored = macro_auc_skip_empty(labels_all[:, texture_idx], student_all[:, texture_idx])

baseline_df = pd.DataFrame([
    {
        'variant': 'teacher_exp015d',
        'macro_auc': teacher_macro_auc,
        'texture_macro_auc': teacher_texture_auc,
        'scored_classes': teacher_scored,
        'texture_scored_classes': teacher_texture_scored,
    },
    {
        'variant': 'student_exp027b',
        'macro_auc': student_macro_auc,
        'texture_macro_auc': student_texture_auc,
        'scored_classes': student_scored,
        'texture_scored_classes': student_texture_scored,
    },
])
display(baseline_df)

sweep_rows: list[dict[str, tp.Any]] = []
for w_student in CFG.weight_grid:
    blend = (1.0 - w_student) * teacher_all + w_student * student_all
    macro_auc, scored_classes = macro_auc_skip_empty(labels_all, blend)
    texture_auc, texture_scored_classes = macro_auc_skip_empty(labels_all[:, texture_idx], blend[:, texture_idx])
    sweep_rows.append({
        'w_student': float(w_student),
        'w_teacher': float(1.0 - w_student),
        'macro_auc': macro_auc,
        'texture_macro_auc': texture_auc,
        'scored_classes': scored_classes,
        'texture_scored_classes': texture_scored_classes,
    })

weight_sweep = pd.DataFrame(sweep_rows).sort_values(['macro_auc', 'texture_macro_auc'], ascending=[False, False]).reset_index(drop=True)
display(weight_sweep.head(12))


,variant,macro_auc,texture_macro_auc,scored_classes,texture_scored_classes
0,teacher_exp015d,0.996472,0.995843,43,30
1,student_exp027b,0.961055,0.958294,43,30


,w_student,w_teacher,macro_auc,texture_macro_auc,scored_classes,texture_scored_classes
0,0.00,1.00,0.996472,0.995843,43,30
1,0.05,0.95,0.994474,0.993062,43,30
2,0.10,0.90,0.993595,0.991896,43,30
3,0.15,0.85,0.992769,0.990747,43,30
4,0.20,0.80,0.992156,0.989857,43,30
5,0.25,0.75,0.991513,0.988988,43,30
6,0.30,0.70,0.990785,0.988149,43,30
7,0.35,0.65,0.989946,0.987201,43,30
8,0.40,0.60,0.989702,0.987297,43,30
9,0.45,0.55,0.989381,0.987177,43,30


In [4]:
taxon_rows: list[dict[str, tp.Any]] = []
taxon_series = taxonomy['class_name'].astype(str)
for taxon in sorted(taxon_series.unique().tolist()):
    idx = np.array([i for i, label in enumerate(classes) if class_name_map.get(label) == taxon], dtype=np.int64)
    if len(idx) == 0:
        continue
    teacher_auc, teacher_sc = macro_auc_skip_empty(labels_all[:, idx], teacher_all[:, idx])
    student_auc, student_sc = macro_auc_skip_empty(labels_all[:, idx], student_all[:, idx])
    best_weight = float(weight_sweep.iloc[0]['w_student'])
    blend = (1.0 - best_weight) * teacher_all[:, idx] + best_weight * student_all[:, idx]
    blend_auc, blend_sc = macro_auc_skip_empty(labels_all[:, idx], blend)
    taxon_rows.append({
        'taxon': taxon,
        'teacher_macro_auc': teacher_auc,
        'student_macro_auc': student_auc,
        'best_blend_macro_auc': blend_auc,
        'teacher_scored_classes': teacher_sc,
        'student_scored_classes': student_sc,
        'blend_scored_classes': blend_sc,
    })

taxon_summary = pd.DataFrame(taxon_rows).sort_values('best_blend_macro_auc', ascending=False).reset_index(drop=True)
display(taxon_summary)

baseline_df.to_csv(OUTPUT_DIR / 'baseline_summary.csv', index=False)
weight_sweep.to_csv(OUTPUT_DIR / 'weight_sweep.csv', index=False)
taxon_summary.to_csv(OUTPUT_DIR / 'taxon_summary.csv', index=False)

report_snapshot = {
    'experiment_id': CFG.experiment_id,
    'experiment_name': CFG.experiment_name,
    'rows': int(len(meta_all)),
    'files': int(meta_all['filename'].nunique()),
    'teacher_macro_auc': float(teacher_macro_auc),
    'student_macro_auc': float(student_macro_auc),
    'teacher_texture_macro_auc': float(teacher_texture_auc),
    'student_texture_macro_auc': float(student_texture_auc),
    'best_weight_student': float(weight_sweep.iloc[0]['w_student']),
    'best_macro_auc': float(weight_sweep.iloc[0]['macro_auc']),
    'best_texture_macro_auc': float(weight_sweep.iloc[0]['texture_macro_auc']),
    'note': 'Teacher scores come from fixed exp_015d replay on fully labeled soundscape rows, not from a fold-safe retraining process.',
}
save_json(report_snapshot, OUTPUT_DIR / 'report_snapshot.json')
print(json.dumps(report_snapshot, indent=2))


,taxon,teacher_macro_auc,student_macro_auc,best_blend_macro_auc,teacher_scored_classes,student_scored_classes,blend_scored_classes
0,Reptilia,1.000000,0.945783,1.000000,1,1,1
1,Aves,0.997901,0.969131,0.997901,11,11,11
2,Amphibia,0.996978,0.923552,0.996978,14,14,14
3,Mammalia,0.996094,0.970313,0.996094,1,1,1
4,Insecta,0.994851,0.988694,0.994851,16,16,16


{
  "experiment_id": "exp_027c",
  "experiment_name": "exp015d_native_student_blend_benchmark",
  "rows": 168,
  "files": 14,
  "teacher_macro_auc": 0.9964723302655292,
  "student_macro_auc": 0.9610550772798211,
  "teacher_texture_macro_auc": 0.9958434292356916,
  "student_texture_macro_auc": 0.9582944669911316,
  "best_weight_student": 0.0,
  "best_macro_auc": 0.9964723302655292,
  "best_texture_macro_auc": 0.9958434292356916,
  "note": "Teacher scores come from fixed exp_015d replay on fully labeled soundscape rows, not from a fold-safe retraining process."
}
